In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import math
import pickle

In [13]:
device = 'mps' if torch.mps.is_available() else 'cpu'
device

'mps'

In [14]:
with open("./transformer_dataset.pkl", 'rb') as f:
    tmp = pickle.load(f)
tmp.keys()

dict_keys(['questions_token', 'answers_token', 'START_TOKEN', 'END_TOKEN', 'VOCAB_SIZE', 'MAX_LENGTH'])

In [15]:
for x in tmp.keys():
    locals()[x] = tmp[x]

In [16]:
def scaled_dot_product_attention(query, key, value, mask=None):
    d_k = key.size(-1)    # 차원 수, 분산을 1로 맞추기 위한 도구
    scores = query @ key.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask, -1e9)
    p_attn = F.softmax(scores, dim=-1)
    return p_attn @ value, p_attn

class ChatbotDataset(Dataset):
    def __init__(self, question, answer):
        self.question = question
        self.answer = answer
    
    def __len__(self):
        return len(self.question)
    
    def __getitem__(self, idx):
        return (
            torch.tensor(self.question[idx], dtype=torch.long),
            torch.tensor(self.answer[idx][:-1], dtype=torch.long),
            torch.tensor(self.answer[idx][1:], dtype=torch.long)
        )

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=9000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        assert d_model % self.num_heads == 0
        self.depth = d_model // self.num_heads

        self.query_dense = nn.Linear(d_model, d_model)
        self.key_dense = nn.Linear(d_model, d_model)
        self.value_dense = nn.Linear(d_model, d_model)
        self.dense = nn.Linear(d_model, d_model)
        
    def split_heads(self, x, batch_size):
        x = x.view(batch_size, -1, self.num_heads, self.depth)
        return x.transpose(1, 2)


    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)


        query = self.query_dense(query)
        key = self.key_dense(key)
        value = self.value_dense(value)


        query = self.split_heads(query, batch_size)
        key = self.split_heads(key, batch_size)
        value = self.split_heads(value, batch_size)


        scaled_attention, _ = scaled_dot_product_attention(query, key, value, mask)
        scaled_attention = scaled_attention.transpose(1, 2).contiguous()
        concat_attention = scaled_attention.view(batch_size, -1, self.d_model)


        return self.dense(concat_attention)

class EncoderLayer(nn.Module):
    def __init__(self, dff, d_model, num_heads, dropout):
        super(EncoderLayer, self).__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = nn.Sequential(       # feed forward network
            nn.Linear(d_model, dff),
            nn.ReLU(),
            nn.Linear(dff, d_model)
        )
        self.layernorm1 = nn.LayerNorm(d_model, eps=1e-6)
        self.layernorm2 = nn.LayerNorm(d_model, eps=1e-6)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)


    def forward(self, x ,  mask):
        attn_output = self.mha(x, x, x, mask)
        attn_output = self.dropout1(attn_output)
        out1 = self.layernorm1(x + attn_output)


        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output)
        out2 = self.layernorm2(out1 + ffn_output)
        return out2


In [17]:
data = ChatbotDataset(questions_token, answers_token)
dataloader = DataLoader(data, batch_size=64, shuffle=True)  

In [18]:
def create_look_ahead_mask(x):
    # (seq_len, seq_len)
    seq_len = x.size(1)
    mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
    return mask.to(device)

def create_padding_mask(x):
    # (batch_size, 1, 1, seq_len)
    return (x == 0).unsqueeze(1).unsqueeze(2)


In [19]:
class EncoderLayer(nn.Module):
    def __init__(self, dff, d_model, num_heads, dropout):
        super(EncoderLayer, self).__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dff),
            nn.ReLU(),
            nn.Linear(dff, d_model)
        )
        self.layernorm1 = nn.LayerNorm(d_model, eps=1e-6)
        self.layernorm2 = nn.LayerNorm(d_model, eps=1e-6)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)


    def forward(self, x ,  mask):
        attn_output = self.mha(x, x, x, mask)
        attn_output = self.dropout1(attn_output)
        out1 = self.layernorm1(x + attn_output)


        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output)
        out2 = self.layernorm2(out1 + ffn_output)
        return out2

class DecoderLayer(nn.Module):
    def __init__(self, dff, d_model, num_heads, dropout):
        super(DecoderLayer, self).__init__()
        self.mha1 = MultiHeadAttention(d_model, num_heads)
        self.mha2 = MultiHeadAttention(d_model, num_heads)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dff),
            nn.ReLU(),
            nn.Linear(dff, d_model)
        )
        self.layernorm1 = nn.LayerNorm(d_model, eps=1e-6)
        self.layernorm2 = nn.LayerNorm(d_model, eps=1e-6)
        self.layernorm3 = nn.LayerNorm(d_model, eps=1e-6)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)
       
    def forward(self, x, enc_output, look_ahead_mask, padding_mask):
        attn1 = self.mha1(x, x, x, look_ahead_mask)
        attn1 = self.dropout1(attn1)
        out1 = self.layernorm1(x + attn1)


        attn2 = self.mha2(out1, enc_output, enc_output, padding_mask)
        attn2 = self.dropout2(attn2)
        out2 = self.layernorm2(out1 + attn2)


        ffn_output = self.ffn(out2)
        ffn_output = self.dropout3(ffn_output)
        out3 = self.layernorm3(out2 + ffn_output)
        return out3



In [20]:
class Transformer(nn.Module):
    def __init__(self, vocab_size, num_layers, dff, d_model, num_heads, dropout):
        super(Transformer, self).__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model)
       
        self.enc_layers = nn.ModuleList([EncoderLayer(dff, d_model, num_heads, dropout) for _ in range(num_layers)])
        self.dec_layers = nn.ModuleList([DecoderLayer(dff, d_model, num_heads, dropout) for _ in range(num_layers)])
        self.dropout = nn.Dropout(dropout)
        self.fc_out = nn.Linear(d_model, vocab_size)
   
    def forward(self, inputs, dec_inputs):
        # 1. 마스크 생성
        enc_padding_mask = create_padding_mask(inputs)
        dec_padding_mask = create_padding_mask(inputs)
        look_ahead_mask = torch.max(
            create_look_ahead_mask(dec_inputs),
            create_padding_mask(dec_inputs)
        )


        # 2. 인코더
        enc_out = self.embedding(inputs) * math.sqrt(self.d_model)
        enc_out = self.dropout(self.pos_encoding(enc_out))
        for layer in self.enc_layers:
            enc_out = layer(enc_out, enc_padding_mask)
       


        # 3. 디코더
        dec_out = self.embedding(dec_inputs) * math.sqrt(self.d_model)
        dec_out = self.dropout(self.pos_encoding(dec_out))
        for layer in self.dec_layers:
            dec_out = layer(dec_out, enc_out, look_ahead_mask, dec_padding_mask)


        return self.fc_out(dec_out)

In [21]:
NUM_LAYERS = 2
D_MODEL = 256
NUM_HEADS = 8
DFF = 512
DROPOUT = 0.1
EPOCHS = 50


class CustomSchedule:
    def __init__(self, d_model, warmup_steps=4000):
        self.d_model = d_model
        self.warmup_steps = warmup_steps


    def __call__(self, step):
        step = step + 1 # 0으로 나누는 것을 방지
        arg1 = step ** -0.5
        arg2 = step * (self.warmup_steps ** -1.5)
        return (self.d_model ** -0.5) * min(arg1, arg2)


model = Transformer(VOCAB_SIZE, NUM_LAYERS, DFF, D_MODEL, NUM_HEADS, DROPOUT).to(device)


criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=CustomSchedule(D_MODEL))


for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch_idx, (inputs, dec_inputs, outputs) in enumerate(dataloader):
        inputs, dec_inputs, outputs = inputs.to(device), dec_inputs.to(device), outputs.to(device)
       
        optimizer.zero_grad()
        predictions = model(inputs, dec_inputs)
       
        # predictions shape: (batch_size, seq_len, vocab_size)
        # outputs shape: (batch_size, seq_len)
        loss = criterion(predictions.view(-1, VOCAB_SIZE), outputs.view(-1))
       
        loss.backward()
        optimizer.step()
        scheduler.step()
       
        total_loss += loss.item()
       
    print(f"Epoch {epoch+1}/{EPOCHS} Loss: {total_loss/len(dataloader):.4f}")


Epoch 1/50 Loss: 8.3363
Epoch 2/50 Loss: 7.1378
Epoch 3/50 Loss: 6.6197
Epoch 4/50 Loss: 6.2695
Epoch 5/50 Loss: 5.8867
Epoch 6/50 Loss: 5.5056
Epoch 7/50 Loss: 5.1275
Epoch 8/50 Loss: 4.7350
Epoch 9/50 Loss: 4.3163
Epoch 10/50 Loss: 3.8815
Epoch 11/50 Loss: 3.4395
Epoch 12/50 Loss: 2.9917
Epoch 13/50 Loss: 2.5597
Epoch 14/50 Loss: 2.1420
Epoch 15/50 Loss: 1.7696
Epoch 16/50 Loss: 1.4124
Epoch 17/50 Loss: 1.1206
Epoch 18/50 Loss: 0.8891
Epoch 19/50 Loss: 0.7178
Epoch 20/50 Loss: 0.6132
Epoch 21/50 Loss: 0.5443
Epoch 22/50 Loss: 0.5053
Epoch 23/50 Loss: 0.4460
Epoch 24/50 Loss: 0.3935
Epoch 25/50 Loss: 0.3554
Epoch 26/50 Loss: 0.3171
Epoch 27/50 Loss: 0.2936
Epoch 28/50 Loss: 0.2704
Epoch 29/50 Loss: 0.2538
Epoch 30/50 Loss: 0.2340
Epoch 31/50 Loss: 0.2124
Epoch 32/50 Loss: 0.2038
Epoch 33/50 Loss: 0.1908
Epoch 34/50 Loss: 0.1784
Epoch 35/50 Loss: 0.1738
Epoch 36/50 Loss: 0.1569
Epoch 37/50 Loss: 0.1537
Epoch 38/50 Loss: 0.1504
Epoch 39/50 Loss: 0.1426
Epoch 40/50 Loss: 0.1328
Epoch 41/

In [22]:
torch.save(model.state_dict(), "./mymodel.pth")